In [5]:
%load_ext autoreload
%autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.append(str(Path().resolve().parents[1]))

In [7]:
import os
import re

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from io import StringIO
import pandas as pd
import pickle
import igraph as ig
import torch



In [ ]:
import torch
import math
import os
import sys
from pathlib import Path
from torch.utils.data import DataLoader
from torch.optim import Adam

DIFFUSION_DIR = str(Path().resolve())
if DIFFUSION_DIR not in sys.path:
    sys.path.insert(0, DIFFUSION_DIR)

from dataset import CachedDataset
from collate import collate_fn
from model import DiffusionGNN
from diff_util import create_diffusion, preprocess

# ----------------------------------
# Config
# ----------------------------------

BATCH_SIZE    = 16
ACCUM_STEPS   = 1
LR            = 2e-4
EPOCHS        = 300
WARMUP_EPOCHS = 15
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

NODE_DIM      = 6
HIDDEN        = 128
TIMESTEPS     = 500
MAX_NODES     = 150
ADJ_LOSS_W    = 0.3
DEG_LOSS_W    = 0.1
LAUND_LOSS_W  = 2.0    # upweight laundering BCE — rare class needs stronger signal
MAX_GRAD_NORM = 1.0

USE_AMP = (DEVICE == "cuda")
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# ----------------------------------
# Dataset
# ----------------------------------

CACHE_PATH = "cached_dataset.pt"

dataset = CachedDataset(CACHE_PATH, max_nodes=MAX_NODES)
print(f"Dataset size: {len(dataset)}")

# ----------------------------------
# Feature normalization stats
# ----------------------------------

all_x = torch.cat([x for x, _ in dataset], dim=0)
x_mean = torch.zeros(NODE_DIM)
x_std  = torch.ones(NODE_DIM)
x_mean[1:] = all_x[:, 1:].mean(0)
x_std[1:]  = all_x[:, 1:].std(0).clamp(min=1e-6)
x_mean = x_mean.to(DEVICE)
x_std  = x_std.to(DEVICE)

laund_rate = all_x[:, 0].mean().item()
print(f"Laundering node rate: {laund_rate:.3f}  (implied pos_weight ≈ {(1-laund_rate)/laund_rate:.1f}x)")
print(f"Feature means (skip 0): {x_mean.tolist()}")
print(f"Feature stds  (skip 0): {x_std.tolist()}")

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)
print(f"Batches per epoch: {len(loader)}  |  Total opt steps: {len(loader) * EPOCHS}")

# ----------------------------------
# Model + diffusion
# ----------------------------------

model = DiffusionGNN(node_dim=NODE_DIM, hidden_dim=HIDDEN, num_layers=4).to(DEVICE)
diffusion = create_diffusion(TIMESTEPS)
print(f"Model on {DEVICE}  |  params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = Adam(model.parameters(), lr=LR)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ----------------------------------
# Training
# ----------------------------------

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, (x, adj, node_mask) in enumerate(loader):

        x         = x.to(DEVICE, dtype=torch.float32)
        adj       = adj.to(DEVICE, dtype=torch.float32)
        node_mask = node_mask.to(DEVICE, dtype=torch.float32)

        x_norm = x.clone()
        x_norm[:, :, 1:] = (x[:, :, 1:] - x_mean[1:]) / x_std[1:]
        x_norm = x_norm * node_mask.unsqueeze(-1)

        adj = adj * node_mask[:, :, None] * node_mask[:, None, :]

        B = x_norm.shape[0]
        t = torch.randint(0, diffusion.num_timesteps, (B,), device=DEVICE)

        with torch.amp.autocast(device_type=DEVICE, enabled=USE_AMP):
            loss_dict = diffusion.training_losses(
                model,
                x_start=x_norm,
                t=t,
                adj_start=adj,
                model_kwargs={"node_mask": node_mask},
                adj_loss_weight=ADJ_LOSS_W,
                deg_loss_weight=DEG_LOSS_W,
                laund_loss_weight=LAUND_LOSS_W,
            )
            loss = loss_dict["loss"].mean() / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

    scheduler.step()

    avg_loss = total_loss / len(loader)
    current_lr = scheduler.get_last_lr()[0]
    if (epoch + 1) % 10 == 0 or epoch < 3:
        print(f"Epoch {epoch+1:4d}/{EPOCHS} | loss={avg_loss:.4f} | lr={current_lr:.2e}")

    torch.save({
        "model": model.state_dict(),
        "x_mean": x_mean.cpu(),
        "x_std": x_std.cpu(),
    }, "model.pt")


In [ ]:

# ============================================================
# Inspection setup — run this cell before the ones below
# ============================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from dataset import CachedDataset
from model import DiffusionGNN
from diff_util import create_diffusion

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
TIMESTEPS = 500
NODE_DIM  = 6
HIDDEN    = 128
MAX_NODES = 150
ADJ_LOSS_W   = 0.3
DEG_LOSS_W   = 0.1
LAUND_LOSS_W = 2.0

FEATURE_NAMES = ["Laundering", "Degree", "Depth", "Betweenness", "Clustering", "PageRank"]

dataset    = CachedDataset("cached_dataset.pt", max_nodes=MAX_NODES)
sample_idx = next(i for i, (x, _) in enumerate(dataset) if x.shape[0] >= 6)
x_orig, adj_orig = dataset[sample_idx]
N = x_orig.shape[0]

print(f"Sample index : {sample_idx}")
print(f"Nodes        : {N}")

checkpoint = torch.load("model.pt", map_location=DEVICE)
x_mean = checkpoint["x_mean"].to(DEVICE)
x_std  = checkpoint["x_std"].to(DEVICE)

x_batch   = x_orig.unsqueeze(0).float().to(DEVICE)
adj_batch = adj_orig.unsqueeze(0).float().to(DEVICE)
node_mask = torch.ones(1, N, device=DEVICE)

x_batch_norm = x_batch.clone()
x_batch_norm[:, :, 1:] = (x_batch[:, :, 1:] - x_mean[1:]) / x_std[1:]

def denorm(x_norm_tensor):
    out = x_norm_tensor.clone()
    out[..., 1:] = x_norm_tensor[..., 1:] * x_std.cpu()[1:] + x_mean.cpu()[1:]
    return out

model = DiffusionGNN(node_dim=NODE_DIM, hidden_dim=HIDDEN, num_layers=4).to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

diffusion = create_diffusion(TIMESTEPS)
print(f"Model loaded on {DEVICE}  |  Laundering nodes in sample: {int(x_orig[:,0].sum())}/{N}")


In [ ]:

# ============================================================
# 1. Forward corruption — q_sample at increasing timesteps
# ============================================================

checkpoints = [0, 125, 250, 375, 499]   # spread across T=500

fig, axes = plt.subplots(len(checkpoints), 1, figsize=(10, 2.5 * len(checkpoints)))

with torch.no_grad():
    for ax, t_val in zip(axes, checkpoints):
        t_tensor = torch.tensor([t_val], device=DEVICE)
        # Pass normalized features — this is the space the model was trained in
        x_t = diffusion.q_sample(x_batch_norm, t_tensor, node_mask=node_mask)
        data = x_t[0].cpu().float().numpy()   # [N, 6]
        im = ax.imshow(data.T, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
        ax.set_title(f"t = {t_val}", fontsize=11)
        ax.set_yticks(range(len(FEATURE_NAMES)))
        ax.set_yticklabels(FEATURE_NAMES)
        ax.set_xlabel("Node index")
        plt.colorbar(im, ax=ax)

plt.suptitle("Forward diffusion: node features at increasing noise levels (normalized space)\n"
             "(feature 0 = Bernoulli corruption, features 1-5 = Gaussian noise)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 2. Encode → decode reconstruction at a chosen t
# ============================================================

def denoise_from_t(model, diffusion, x_t, adj_t, node_mask, start_t, device):
    """Run the reverse diffusion loop from `start_t` down to 0."""
    x   = x_t.clone()
    adj = adj_t.clone()
    with torch.no_grad():
        for i in reversed(range(start_t + 1)):
            t = torch.tensor([i] * x.shape[0], device=device)
            out = diffusion.p_sample(
                model, x, t,
                model_kwargs={"adj": adj, "node_mask": node_mask},
            )
            x   = out["sample"]
            adj = out["adj_sample"]
    return x, adj


T_ENCODE = 50   # try 25, 50, 75, 250

with torch.no_grad():
    t_tensor = torch.tensor([T_ENCODE], device=DEVICE)
    # Corrupt in normalized space (same as training)
    x_noisy_norm, adj_noisy = diffusion.q_sample(
        x_batch_norm, t_tensor, node_mask=node_mask, adj_start=adj_batch
    )

x_recon_norm, adj_recon = denoise_from_t(
    model, diffusion, x_noisy_norm, adj_noisy, node_mask, T_ENCODE, DEVICE
)

# Denormalize for display — convert model-space outputs back to original feature scale
x_orig_display  = denorm(x_batch_norm[0].cpu())
x_noisy_display = denorm(x_noisy_norm[0].cpu())
x_recon_display = denorm(x_recon_norm[0].cpu())

# ---- Node feature plot (denormalized = original scale) ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, title, data in zip(
    axes,
    ["Original", f"Noisy (t={T_ENCODE})", "Reconstructed"],
    [x_orig_display, x_noisy_display, x_recon_display],
):
    im = ax.imshow(data.T.numpy(), aspect="auto", cmap="RdBu_r")
    ax.set_title(title, fontsize=12)
    ax.set_yticks(range(len(FEATURE_NAMES)))
    ax.set_yticklabels(FEATURE_NAMES)
    ax.set_xlabel("Node index")
    plt.colorbar(im, ax=ax)

# MSE in normalized space (fair comparison — same scale as training loss)
mse_norm = ((x_batch_norm - x_recon_norm) ** 2).mean().item()
plt.suptitle(f"Encode → decode (t={T_ENCODE})  |  Node MSE (norm. space): {mse_norm:.4f}", fontsize=13)
plt.tight_layout()
plt.show()

# ---- Adjacency plot ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, title, mat in zip(
    axes,
    ["Original adj", f"Noisy adj (t={T_ENCODE})", "Reconstructed adj"],
    [adj_batch, adj_noisy, adj_recon],
):
    ax.imshow(mat[0].cpu().float().numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

adj_acc = ((adj_recon[0] > 0.5).float() == adj_batch[0]).float().mean().item()
plt.suptitle(f"Adjacency reconstruction  |  Edge accuracy: {adj_acc:.2%}", fontsize=13)
plt.tight_layout()
plt.show()

# Per-feature MSE (normalized space) + laundering accuracy
print("Per-feature reconstruction MSE (normalized space):")
for i, name in enumerate(FEATURE_NAMES):
    feat_mse = ((x_batch_norm[0, :, i] - x_recon_norm[0, :, i]) ** 2).mean().item()
    print(f"  {name:12s} MSE: {feat_mse:.4f}")

orig_bin  = (x_orig[:, 0].numpy() > 0.5)
recon_bin = (x_recon_norm[0, :, 0].cpu().float().numpy() > 0.5)
print(f"\nLaundering label accuracy: {(orig_bin == recon_bin).mean():.2%}")


In [ ]:

# ============================================================
# 3. Full generation from pure noise (p_sample_loop)
# ============================================================

shape     = x_batch_norm.shape   # [1, N, 6]
adj_shape = adj_batch.shape      # [1, N, N]

with torch.no_grad():
    x_generated_norm, adj_generated = diffusion.p_sample_loop(
        model,
        shape,
        adj_shape=adj_shape,
        model_kwargs={"node_mask": node_mask},
        device=DEVICE,
    )

# Denormalize generated features for display in original scale
x_generated_display = denorm(x_generated_norm[0].cpu())
x_orig_display      = denorm(x_batch_norm[0].cpu())

# ---- Node feature comparison (original scale) ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, data in zip(
    axes,
    ["Original node features", "Generated node features"],
    [x_orig_display, x_generated_display],
):
    im = ax.imshow(data.T.numpy(), aspect="auto", cmap="RdBu_r")
    ax.set_title(title, fontsize=12)
    ax.set_yticks(range(len(FEATURE_NAMES)))
    ax.set_yticklabels(FEATURE_NAMES)
    ax.set_xlabel("Node index")
    plt.colorbar(im, ax=ax)
plt.suptitle("Full generation: node features (original scale)", fontsize=13)
plt.tight_layout()
plt.show()

# ---- Adjacency comparison ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, mat in zip(
    axes,
    ["Original adj", "Generated adj"],
    [adj_batch, (adj_generated > 0.5).float()],
):
    ax.imshow(mat[0].cpu().float().numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
plt.suptitle("Full generation: adjacency matrix", fontsize=13)
plt.tight_layout()
plt.show()

# ---- Stats (original scale, denormalized) ----
orig_np = x_orig_display.numpy()
gen_np  = x_generated_display.numpy()
print(f"{'Feature':<14} {'orig mean':>10} {'gen mean':>10} {'orig std':>10} {'gen std':>10}")
for i, name in enumerate(FEATURE_NAMES):
    print(f"{name:<14} {orig_np[:, i].mean():>10.3f} {gen_np[:, i].mean():>10.3f} "
          f"{orig_np[:, i].std():>10.3f} {gen_np[:, i].std():>10.3f}")

orig_density = adj_batch[0].mean().item()
gen_density  = (adj_generated[0] > 0.5).float().mean().item()
print(f"\nEdge density — original: {orig_density:.3f}  generated: {gen_density:.3f}")


In [ ]:

# ============================================================
# 4. Per-timestep loss curve
# ============================================================

K = 8

losses_per_t = []

model.eval()
with torch.no_grad():
    for t_val in range(TIMESTEPS):
        t_tensor = torch.tensor([t_val], device=DEVICE)
        k_losses = []
        for _ in range(K):
            loss_dict = diffusion.training_losses(
                model,
                x_start=x_batch_norm,
                t=t_tensor,
                adj_start=adj_batch,
                model_kwargs={"node_mask": node_mask},
                adj_loss_weight=ADJ_LOSS_W,
                deg_loss_weight=DEG_LOSS_W,
                laund_loss_weight=LAUND_LOSS_W,
            )
            k_losses.append(loss_dict["loss"].item())
        losses_per_t.append(sum(k_losses) / K)

window = max(1, TIMESTEPS // 20)
smoothed = np.convolve(losses_per_t, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(range(TIMESTEPS), losses_per_t, alpha=0.35, linewidth=1,
         color="steelblue", label=f"Loss per t (avg {K} samples)")
plt.plot(range(window - 1, TIMESTEPS), smoothed, linewidth=2,
         color="red", label=f"Smoothed (w={window})")
plt.xlabel("Timestep t")
plt.ylabel("Combined loss")
plt.title(f"Denoising loss vs timestep (K={K} samples/t)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Easiest t: {int(np.argmin(losses_per_t))}  loss={min(losses_per_t):.4f}")
print(f"Hardest t: {int(np.argmax(losses_per_t))}  loss={max(losses_per_t):.4f}")


In [ ]:

# ============================================================
# 5. Graph visualization — original vs reconstructed laundering
# ============================================================
# Requires Cell 2 (defines denoise_from_t) to have been run first.

T_VIZ = 50

with torch.no_grad():
    t_tensor = torch.tensor([T_VIZ], device=DEVICE)
    x_noisy_viz, adj_noisy_viz = diffusion.q_sample(
        x_batch_norm, t_tensor, node_mask=node_mask, adj_start=adj_batch
    )

x_recon_viz, adj_recon_viz = denoise_from_t(
    model, diffusion, x_noisy_viz, adj_noisy_viz, node_mask, T_VIZ, DEVICE
)

adj_np      = adj_orig.cpu().numpy()
G           = nx.from_numpy_array(adj_np)
pos         = nx.spring_layout(G, seed=42)

orig_labels  = x_orig[:, 0].numpy()                               # raw binary labels
recon_labels = x_recon_viz[0, :, 0].cpu().float().numpy()         # feature 0 not normalized
recon_binary = (recon_labels > 0.5).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, labels, title in zip(
    axes,
    [orig_labels, recon_binary],
    ["Original laundering labels", f"Reconstructed (encode t={T_VIZ} → decode)"],
):
    colors = ["red" if l > 0.5 else "steelblue" for l in labels]
    nx.draw_networkx(
        G, pos=pos, ax=ax,
        node_color=colors, node_size=400,
        with_labels=True, font_size=7,
        edge_color="grey", alpha=0.85,
    )
    ax.set_title(title, fontsize=12)
    ax.axis("off")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="red",       label="Laundering"),
    Patch(facecolor="steelblue", label="Clean"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("Graph: laundering node detection (original vs reconstructed)", fontsize=13)
plt.tight_layout()
plt.show()

correct = ((recon_binary > 0.5) == (orig_labels > 0.5)).mean()
print(f"Laundering label accuracy after encode-decode: {correct:.2%}")
